abbind也应该是预测ddg会用到的benchmark，但没啥可以分析的，详见scripts

### 1. Five DMS dataset in science

实际上是四个，第五个只是单点

In [1]:
import os
import pandas as pd
import numpy as np
from Bio import SeqIO

#### 1.1 获取并检查抗原序列

In [2]:
# 复制序列
h1a1_seq_ncbi = "MKAKLLVLLCTFTATYADTICIGYHANNSTDTVDTVLEKNVTVTHSVNLLEDSHNGKLCLLKGIAPLQLGNCSVAGWILGNPECELLISKESWSYIVETPNPENGTCYPGYFADYEELREQLSSVSSFERFEIFPKESSWPNHTVTGVSASCSHNGKSSFYRNLLWLTGKNGLYPNLSKSYVNNKEKEVLVLWGVHHPPNIGNQRALYHTENAYVSVVSSHYSRRFTPEIAKRPKVRDQEGRINYYWTLLEPGDTIIFEANGNLIAPWYAFALSRGFGSGIITSNAPMDECDAKCQTPQGAINSSLPFQNVHPVTIGECPKYVRSAKLRMVTGLRNIPSIQSR"
h1a2_seq_ncbi = "GLFGAIAGFIEGGWTGMVDGWYGYHHQNEQGSGYAADQKSTQNAINGITNKVNSVIEKMNTQFTAVGKEFNKLERRMENLNKKVDDGFLDIWTYNAELLVLLENERTLDFHDSNVKNLYEKVKSQLKNNAKEIGNGCFEFYHKCNNECMESVKNGTYDYPKYSEESKLNREKIDGVKLESMGVYQILAIYSTVASSLVLLVSLGAISFWMCSNGSLQCRICI"
h1a1_seq_plasmid = "DTICIGYHANNSTDTVDTVLEKNVTVTHSVNLLEDSHNGKLCLLKGIAPLQLGNCSVAGWILGNPECELLISKESWSYIVETPNPENGTCYPGYFADYEELREQLSSVSSFERFEIFPKESSWPNHTVTGVSASCSHNGKSSFYRNLLWLTGKNGLYPNLSKSYVNNKEKEVLVLWGVHHPPNIGNQRALYHTENAYVSVVSSHYSRRFTPEIAKRPKVRDQEGRINYYWTLLEPGDTIIFEANGNLIAPWYAFALSRGFGSGIITSNAPMDECDAKCQTPQGAINSSLPFQNVHPVTIGECPKYVRSAKLRMVTGLRNIPSIQSR"
h1a2_seq_plasmid = "GLFGAIAGFIEGGWTGMVDGWYGYHHQNEQGSGYAADQKSTQNAINGITNKVNSVIEKMNTQFTAVGKEFNKLERRMENLNKKVDDGFLDIWTYNAELLVLLENERTLDFHDSNVKNLYEKVKSQLKNNAKEIGNGCFEFYHKCNNECMESVKNGTYDYPKYSEESKLNREKIDGVSGGGGLNDIFEAQKIEWHERLVPRGSPGSGYIPEAPRDGQAYVRKDGEWVLLSTFLGHHHHHH"

h3a1_seq_ncbi = "QKLPGNDNSTATLCLGHHAVPNGTIVKTITNDQIEVTNATELVQSSSTGGICDSPHQILDGENCTLIDALLGDPQCDGFQNKKWDLFVERSKAYSNCYPYDVPDYASLRSLVASSGTLEFNDESFNWTGVTQNGTSSSCKRRSNNSFFSRLNWLTHLKFKYPALNVTMPNNEKFDKLYIWGVHHPVTDNDQIFLYAQASGRITVSTKRSQQTVIPNIGSRPRIRNIPSRISIYWTIVKPGDILLINSTGNLIAPRGYFKIRSGKSSIMRSDAPIGKCNSECITPNGSIPNDKPFQNVNRITYGACPRYVKQNTLKLATGMRNVPEKQTRGIFGAIAG"
h3a2_seq_ncbi = "FIENGWEGMVDGWYGFRHQNSEGIGQAADLKSTQAAINQINGKLNRLIGKTNEKFHQIEKEFSEVEGRIQDLEKYVEDTKIDLWSYNAELLVALENQHTIDLTDSEMNKLFERTKKQLRENAEDMGNGCFKIYHKCDNACIGSIRNGTYDHDVYRDEALNNRFQIKGVELKSGYKDWILWISFAISCFLLCVALLGFIMWACQKGNIRCNICI"
h3a1_seq_plasmid = "QKLPGNDNSTATLCLGHHAVPNGTIVKTITNDQIEVTNATELVQSSSTGGICDSPHQILDGENCTLIDALLGDPQCDGFQNKKWDLFVERSKAYSNCYPYDVPDYASLRSLVASSGTLEFNDESFNWTGVTQNGTSSSCKRRSNNSFFSRLNWLTHLKFKYPALNVTMPNNEKFDKLYIWGVHHPVTDNDQIFLYAQASGRITVSTKRSQQTVIPNIGSRPRIRNIPSRISIYWTIVKPGDILLINSTGNLIAPRGYFKIRSGKSSIMRSDAPIGKCNSECITPNGSIPNDKPFQNVNRITYGACPRYVKQNTLKLATGMRNVPEKQTRGIFGAIAG"
h3a2_seq_plasmid = "FIENGWEGMVDGWYGFRHQNSEGIGQAADLKSTQAAINQINGKLNRLIGKTNEKFHQIEKEFSEVEGRIQDLEKYVEDTKIDLWSYNAELLVALENQHTIDLTDSEMNKLFERTKKQLRENAEDMGNGCFKIYHKCDNACIGSIRNGTYDHDVYRDEALNNRFQIKGVSGGGGLNDIFEAQKIEWHERLVPRGSPGSGYIPEAPRDGQAYVRKDGEWVLLSTFL"

h9a1_seq_ncbi = "METISLITILLVVTASNADKICIGHQSTNSTETVDTLTETNVPVTHAKELLHTEHNGMLCATSLGHPLILDTCTIEGLVYGNPSCDLLLGGREWSYIVERSSAVNGTCYPGNVENLEELRTLFSSASSYQRIQIFPDTTWNVTYTGTSRACSGSFYRSMRWLTQKSGFYPVQDAQYTNNRGKSILFVWGIHHPPTYTEQTNLYIRNDTTTSVTTEDLNRTFKPVIGPRPLVNGLQGRIDYYWSVLKPGQTLRVRSNGNLIAPWYGHVLSGGSHGRILKTDLKGGNCVVQCQTEKGGLNSTLPFHNISKYAFGTCPKYVRVNSLKLAVGLRNVPARSSR"
h9a2_seq_ncbi = "GLFGAIAGFIEGGWPGLVAGWYGFQHSNDQGVGMAADRDSTQKAIDKITSKVNNIVDKMNKQYEIIDHEFSEVETRLNMINNKIDDQIQDVWAYNAELLVLLENQKTLDEHDANVNNLYNKVKRALGSNAMEDGKGCFELYHKCDDQCMETIRNGTYNRRKYREESRLERQKIEGVKLESEGTYKILTIYSTVASSLVLAMGFAAFLFWAMSNGSCRCNICI"
h9a1_seq_plasmid = "ADPGADKICIGHQSTNSTETVDTLTETNVPVTHAKELLHTEHNGMLCATSLGHPLILDTCTIEGLVYGNPSCDLLLGGREWSYIVERSSAVNGTCYPGNVENLEELRTLFSSASSYQRIQIFPDTTWNVTYTGTSRACSGSFYRSMRWLTQKSGFYPVQDAQYTNNRGKSILFVWGIHHPPTYTEQTNLYIRNDTTTSVTTEDLNRTFKPVIGPRPLVNGLQGRIDYYWSVLKPGQTLRVRSNGNLIAPWYGHVLSGGSHGRILKTDLKGGNCVVQCQTEKGGLNSTLPFHNISKYAFGTCPKYVRVNSLKLAVGLRNVPARSSR"
h9a2_seq_plasmid = "GLFGAIAGFIEGGWPGLVAGWYGFQHSNDQGVGMAADRDSTQKAIDKITSKVNNIVDKMNKQYEIIDHEFSEVETRLNMINNKIDDQIQDVWAYNAELLVLLENQKTLDEHDANVNNLYNKVKRALGSNAMEDGKGCFELYHKCDDQCMETIRNGTYNRRKYREESRLERQKIEGVKLESESGGGGLNDIFEAQKIEWHERLVPRGSPGSGYIPEAPRDGQAYVRKDGEWVLLSTFLGHHHHHH"

# 取交集--质粒只表达了病毒蛋白的一部分，根据观察结果，手动去掉头尾不同的，然后判断中间是否相同
h1a1_seq_ncbi_new = "DTICIGYHANNSTDTVDTVLEKNVTVTHSVNLLEDSHNGKLCLLKGIAPLQLGNCSVAGWILGNPECELLISKESWSYIVETPNPENGTCYPGYFADYEELREQLSSVSSFERFEIFPKESSWPNHTVTGVSASCSHNGKSSFYRNLLWLTGKNGLYPNLSKSYVNNKEKEVLVLWGVHHPPNIGNQRALYHTENAYVSVVSSHYSRRFTPEIAKRPKVRDQEGRINYYWTLLEPGDTIIFEANGNLIAPWYAFALSRGFGSGIITSNAPMDECDAKCQTPQGAINSSLPFQNVHPVTIGECPKYVRSAKLRMVTGLRNIPSIQSR"
h1a1_seq_plasmid_new = h1a1_seq_plasmid

h1a2_seq_ncbi_new = "GLFGAIAGFIEGGWTGMVDGWYGYHHQNEQGSGYAADQKSTQNAINGITNKVNSVIEKMNTQFTAVGKEFNKLERRMENLNKKVDDGFLDIWTYNAELLVLLENERTLDFHDSNVKNLYEKVKSQLKNNAKEIGNGCFEFYHKCNNECMESVKNGTYDYPKYSEESKLNREKIDGV"
h1a2_seq_plasmid_new = "GLFGAIAGFIEGGWTGMVDGWYGYHHQNEQGSGYAADQKSTQNAINGITNKVNSVIEKMNTQFTAVGKEFNKLERRMENLNKKVDDGFLDIWTYNAELLVLLENERTLDFHDSNVKNLYEKVKSQLKNNAKEIGNGCFEFYHKCNNECMESVKNGTYDYPKYSEESKLNREKIDGV"

h3a1_seq_ncbi_new = h3a1_seq_ncbi
h3a1_seq_plasmid_new = h3a1_seq_plasmid

h3a2_seq_ncbi_new = "FIENGWEGMVDGWYGFRHQNSEGIGQAADLKSTQAAINQINGKLNRLIGKTNEKFHQIEKEFSEVEGRIQDLEKYVEDTKIDLWSYNAELLVALENQHTIDLTDSEMNKLFERTKKQLRENAEDMGNGCFKIYHKCDNACIGSIRNGTYDHDVYRDEALNNRFQIKGV"
h3a2_seq_plasmid_new = "FIENGWEGMVDGWYGFRHQNSEGIGQAADLKSTQAAINQINGKLNRLIGKTNEKFHQIEKEFSEVEGRIQDLEKYVEDTKIDLWSYNAELLVALENQHTIDLTDSEMNKLFERTKKQLRENAEDMGNGCFKIYHKCDNACIGSIRNGTYDHDVYRDEALNNRFQIKGV"

h9a1_seq_ncbi_new = "ADKICIGHQSTNSTETVDTLTETNVPVTHAKELLHTEHNGMLCATSLGHPLILDTCTIEGLVYGNPSCDLLLGGREWSYIVERSSAVNGTCYPGNVENLEELRTLFSSASSYQRIQIFPDTTWNVTYTGTSRACSGSFYRSMRWLTQKSGFYPVQDAQYTNNRGKSILFVWGIHHPPTYTEQTNLYIRNDTTTSVTTEDLNRTFKPVIGPRPLVNGLQGRIDYYWSVLKPGQTLRVRSNGNLIAPWYGHVLSGGSHGRILKTDLKGGNCVVQCQTEKGGLNSTLPFHNISKYAFGTCPKYVRVNSLKLAVGLRNVPARSSR"
h9a1_seq_plasmid_new = "ADKICIGHQSTNSTETVDTLTETNVPVTHAKELLHTEHNGMLCATSLGHPLILDTCTIEGLVYGNPSCDLLLGGREWSYIVERSSAVNGTCYPGNVENLEELRTLFSSASSYQRIQIFPDTTWNVTYTGTSRACSGSFYRSMRWLTQKSGFYPVQDAQYTNNRGKSILFVWGIHHPPTYTEQTNLYIRNDTTTSVTTEDLNRTFKPVIGPRPLVNGLQGRIDYYWSVLKPGQTLRVRSNGNLIAPWYGHVLSGGSHGRILKTDLKGGNCVVQCQTEKGGLNSTLPFHNISKYAFGTCPKYVRVNSLKLAVGLRNVPARSSR"

h9a2_seq_ncbi_new = "GLFGAIAGFIEGGWPGLVAGWYGFQHSNDQGVGMAADRDSTQKAIDKITSKVNNIVDKMNKQYEIIDHEFSEVETRLNMINNKIDDQIQDVWAYNAELLVLLENQKTLDEHDANVNNLYNKVKRALGSNAMEDGKGCFELYHKCDDQCMETIRNGTYNRRKYREESRLERQKIEGVKLESE"
h9a2_seq_plasmid_new = "GLFGAIAGFIEGGWPGLVAGWYGFQHSNDQGVGMAADRDSTQKAIDKITSKVNNIVDKMNKQYEIIDHEFSEVETRLNMINNKIDDQIQDVWAYNAELLVLLENQKTLDEHDANVNNLYNKVKRALGSNAMEDGKGCFELYHKCDDQCMETIRNGTYNRRKYREESRLERQKIEGVKLESE"

# 检查ncbi序列和质粒中疑似序列是否完全匹配
print("H1a1 seq match:", h1a1_seq_ncbi_new == h1a1_seq_plasmid_new)
print("H1a2 seq match:", h1a2_seq_ncbi_new == h1a2_seq_plasmid_new)
print("H3a1 seq match:", h3a1_seq_ncbi_new == h3a1_seq_plasmid_new)
print("H3a2 seq match:", h3a2_seq_ncbi_new == h3a2_seq_plasmid_new)
print("H9a1 seq match:", h9a1_seq_ncbi_new == h9a1_seq_plasmid_new)
print("H9a2 seq match:", h9a2_seq_ncbi_new == h9a2_seq_plasmid_new)

H1a1 seq match: True
H1a2 seq match: True
H3a1 seq match: True
H3a2 seq match: True
H9a1 seq match: True
H9a2 seq match: True


In [22]:
# 检查ncbi序列是否和pdb序列有相关性
h1a1_pdb_seq = "ADPGDTICIGYHANNSTDTVDTVLEKNVTVTHSVNLLEDSHNGKLCKLKGIAPLQLGKCNIAGWLLGNPECDLLLTASSWSYIVETSNSENGTCYPGDFIDYEELREQLSSVSSFEKFEIFPKTSSWPNHETTKGVTAACSYAGASSFYRNLLWLTKKGSSYPKLSKSYVNNKGKEVLVLWGVHHPPTGTDQQSLYQNADAYVSVGSSKYNRRFTPEIAARPKVRDQAGRMNYYWTLLEPGDTITFEATGNLIAPWYAFALNRGSGSGIITSDAPVHDCNTKCQTPHGAINSSLPFQNIHPVTIGECPKYVRSTKLRMATGLRNIPSIQSR"
h1a2_pdb_seq = "GLFGAIAGFIEGGWTGMIDGWYGYHHQNEQGSGYAADQKSTQNAIDGITNKVNSVIEKMNTQFTAVGKEFNNLERRIENLNKKVDDGFLDIWTYNAELLVLLENERTLDFHDSNVRNLYEKVKSQLKNNAKEIGNGCFEFYHKCDDACMESVRNGTYDYPKYSEESKLNREEIDGVSGR"

h5a1_pdb_seq = "ADPGDQICIGYHANNSTEQVDTIMEKNVTVTHAQDILEKKHNGKLCDLDGVKPLILRDCSVAGWLLGNPMCDEFINVPEWSYIVEKANPVNDLCYPGDFNDYEELKHLLSRINHFEKIQIIPKSSWSSHEASLGVSSACPYQGKSSFFRNVVWLIKKNSTYPTIKRSYNNTNQEDLLVLWGIHHPNDAAEQTKLYQNPTTYISVGTSTLNQRLVPRIATRSKVNGQSGRMEFFWTILKPNDAINFESNGNFIAPEYAYKIVKKGDSTIMKSELEYGNCNTKCQTPMGAINSSMPFHNIHPLTIGECPKYVKSNRLVLATGLRNSPQRERRRKKR"
h5a2_pdb_seq = "GLFGAIAGFIEGGWQGMVDGWYGYHHSNEQGSGYAADKESTQKAIDGVTNKVNSIIDKMNTQFEAVGREFNNLERRIENLNKKMEDGFLDVWTYNAELLVLMENERTLDFHDSNVKNLYDKVRLQLRDNAKELGNGCFEFYHKCDNECMESVRNGTYDYPQYSEEARLKREEISSGR"

print(h1a1_pdb_seq)
print(h9a1_seq_ncbi_new)
print(h5a1_pdb_seq)
print('\n')
print(h1a2_pdb_seq)
print(h9a2_seq_ncbi_new)
print(h5a2_pdb_seq)

ADPGDTICIGYHANNSTDTVDTVLEKNVTVTHSVNLLEDSHNGKLCKLKGIAPLQLGKCNIAGWLLGNPECDLLLTASSWSYIVETSNSENGTCYPGDFIDYEELREQLSSVSSFEKFEIFPKTSSWPNHETTKGVTAACSYAGASSFYRNLLWLTKKGSSYPKLSKSYVNNKGKEVLVLWGVHHPPTGTDQQSLYQNADAYVSVGSSKYNRRFTPEIAARPKVRDQAGRMNYYWTLLEPGDTITFEATGNLIAPWYAFALNRGSGSGIITSDAPVHDCNTKCQTPHGAINSSLPFQNIHPVTIGECPKYVRSTKLRMATGLRNIPSIQSR
ADKICIGHQSTNSTETVDTLTETNVPVTHAKELLHTEHNGMLCATSLGHPLILDTCTIEGLVYGNPSCDLLLGGREWSYIVERSSAVNGTCYPGNVENLEELRTLFSSASSYQRIQIFPDTTWNVTYTGTSRACSGSFYRSMRWLTQKSGFYPVQDAQYTNNRGKSILFVWGIHHPPTYTEQTNLYIRNDTTTSVTTEDLNRTFKPVIGPRPLVNGLQGRIDYYWSVLKPGQTLRVRSNGNLIAPWYGHVLSGGSHGRILKTDLKGGNCVVQCQTEKGGLNSTLPFHNISKYAFGTCPKYVRVNSLKLAVGLRNVPARSSR
ADPGDQICIGYHANNSTEQVDTIMEKNVTVTHAQDILEKKHNGKLCDLDGVKPLILRDCSVAGWLLGNPMCDEFINVPEWSYIVEKANPVNDLCYPGDFNDYEELKHLLSRINHFEKIQIIPKSSWSSHEASLGVSSACPYQGKSSFFRNVVWLIKKNSTYPTIKRSYNNTNQEDLLVLWGIHHPNDAAEQTKLYQNPTTYISVGTSTLNQRLVPRIATRSKVNGQSGRMEFFWTILKPNDAINFESNGNFIAPEYAYKIVKKGDSTIMKSELEYGNCNTKCQTPMGAINSSMPFHNIHPLTIGECPKYVKSNRLVLATGLRNSPQRERRRKKR


GLFGAIAGF

In [35]:
# 检查ncbi序列和science附表

# 先检查pdb序列是否和science附表的epitop对上了，HA1利用40 41 42连号和291 292 293连号进行检查，HA2利用19 20 21连号检查
# print(h1a1_pdb_seq)
# print(h1a2_pdb_seq)
# print(h5a1_pdb_seq)
# print(h5a2_pdb_seq)

print(h1a1_pdb_seq.find('VNL'))  # 38 -> 33
print(h1a1_pdb_seq.find('SLP'))  # 291 -> 292
print(h1a2_pdb_seq.find('DGW'))  # 19 -> 18

print(h5a1_pdb_seq.find('QDI'))  # 38 -> 33
print(h5a1_pdb_seq.find('SMP'))  # 291 -> 291
print(h5a2_pdb_seq.find('VDGW'))  # 19 -> 17

# 再看自己查的和science附表标注的epitop是否一致
print(h1a1_seq_ncbi_new.find('VNL'))
print(h1a1_seq_ncbi_new.find('SLP'))
print(h1a2_seq_ncbi_new.find('VDGW'))

print(h3a1_seq_ncbi_new.find('TEL'))
print(h3a1_seq_ncbi_new.find('DKP'))
print(h3a2_seq_ncbi_new.find('VDGW'))

print(h9a1_seq_ncbi_new.find('KEL'))
print(h9a1_seq_ncbi_new.find('TLP'))
print(h9a2_seq_ncbi_new.find('AGW'))

33
292
18
33
291
17
29
287
17
39
290
9
30
282
18


#### 1.2 获取并检查抗体序列

In [7]:
# 注意是对应elife论文中的somatic，要和pdb中序列对比检查（没啥差别，轻链重链都直接复制pdb里面的吧！）

CR9114_h_seq_somatic = "QVQLVQSGAEVKKPGSSVKVSCKSSGGTSNNYAISWVRQAPGQGLDWMGGISPIFGSTAYAQKFQGRVTISADIFSNTAYMELNSLTSEDTAVYFCARHGNYYYYSGMDVWGQGTTVTVSS"
CR6261_h_seq_somatic = "QVQLVESGAEVKKPGSSVKVSCKASGGPFRSYAISWVRQAPGQGPEWMGGIIPIFGTTKYAPKFQGRVTITADDFAGTYYMELSSLRSEDTAMYYCAKHMGYQYRETMDVWGQGTTVTVSS"

CR9114_h_seq_genbank = ""
CR6261_h_seq_genbank = ""
CR9114_l_seq_genbank = "SYVLTQPPAVSGTPGQRVTISCSGSDSNIGRRSVNWYQQFPGTAPKLLIYSNDQRPSVVPDRFSGSKSGTSASLAISGLQSEDEAEYYCAAWDDSLKGAVFGGGTQLTVL"
CR6261_l_seq_genbank = ""

CR9114_h_seq_pdb = "QVQLVQSGAEVKKPGSSVKVSCKSSGGTSNNYAISWVRQAPGQGLDWMGGISPIFGSTAYAQKFQGRVTISADIFSNTAYMELNSLTSEDTAVYFCARHGNYYYYSGMDVWGQGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVEPKSCHHHHHH"
CR9114_l_seq_pdb = "QSALTQPPAVSGTPGQRVTISCSGSDSNIGRRSVNWYQQFPGTAPKLLIYSNDQRPSVVPDRFSGSKSGTSASLAISGLQSEDEAEYYCAAWDDSLKGAVFGGGTQLTVLGQPKAAPSVTLFPPSSEELQANKATLVCLISDFYPGAVTVAWKADSSPVKAGVETTTPSKQSNNKYAASSYLSLTPEQWKSHRSYSCQVTHEGSTVEKTVAPTECS"
CR6261_h_seq_pdb = "EVQLVESGAEVKKPGSSVKVSCKASGGPFRSYAISWVRQAPGQGPEWMGGIIPIFGTTKYAPKFQGRVTITADDFAGTVYMELSSLRSEDTAMYYCAKHMGYQVRETMDVWGKGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVEPKSCDK"
CR6261_l_seq_pdb = "QSVLTQPPSVSAAPGQKVTISCSGSSSNIGNDYVSWYQQLPGTAPKLLIYDNNKRPSGIPDRFSGSKSGTSATLGITGLQTGDEANYYCATWDRRPTAYVVFGGGTKLTVLGAAAGQPKAAPSVTLFPPSSEELQANKATLVCLISDFYPGAVTVAWKADSSPVKAGVETTTPSKQSNNKYAASSYLSLTPEQWKSHRSYSCQVTHEGSTVEKTVAPTECS"

print(CR9114_h_seq_somatic)
print(CR9114_h_seq_pdb)

print('\n')

print(CR6261_h_seq_somatic)
print(CR6261_h_seq_pdb)


QVQLVQSGAEVKKPGSSVKVSCKSSGGTSNNYAISWVRQAPGQGLDWMGGISPIFGSTAYAQKFQGRVTISADIFSNTAYMELNSLTSEDTAVYFCARHGNYYYYSGMDVWGQGTTVTVSS
QVQLVQSGAEVKKPGSSVKVSCKSSGGTSNNYAISWVRQAPGQGLDWMGGISPIFGSTAYAQKFQGRVTISADIFSNTAYMELNSLTSEDTAVYFCARHGNYYYYSGMDVWGQGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVEPKSCHHHHHH


QVQLVESGAEVKKPGSSVKVSCKASGGPFRSYAISWVRQAPGQGPEWMGGIIPIFGTTKYAPKFQGRVTITADDFAGTYYMELSSLRSEDTAMYYCAKHMGYQYRETMDVWGQGTTVTVSS
EVQLVESGAEVKKPGSSVKVSCKASGGPFRSYAISWVRQAPGQGPEWMGGIIPIFGTTKYAPKFQGRVTITADDFAGTVYMELSSLRSEDTAMYYCAKHMGYQVRETMDVWGKGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVEPKSCDK


In [9]:
print("EVQLVESGAEVKKPGSSVKVSCKASGGPFRSYAISWVRQAPGQGPEWMGGIIPIFGTTKYAPKFQGRVTITADDFAGTVYMELSSLRSEDTAMYYCAKHMGYQVRETMDVWGKGTTVTVSS")
print("QVQLVESGAEVKKPGSSVKVSCKASGGPFRSYAISWVRQAPGQGPEWMGGIIPIFGTTKYAPKFQGRVTITADDFAGTYYMELSSLRSEDTAMYYCAKHMGYQYRETMDVWGQGTTVTVSS")

EVQLVESGAEVKKPGSSVKVSCKASGGPFRSYAISWVRQAPGQGPEWMGGIIPIFGTTKYAPKFQGRVTITADDFAGTVYMELSSLRSEDTAMYYCAKHMGYQVRETMDVWGKGTTVTVSS
QVQLVESGAEVKKPGSSVKVSCKASGGPFRSYAISWVRQAPGQGPEWMGGIIPIFGTTKYAPKFQGRVTITADDFAGTYYMELSSLRSEDTAMYYCAKHMGYQYRETMDVWGQGTTVTVSS


#### 1.3 检查csv突变data

In [3]:
cr6261_data_path = "/home/dataset-local/projects_dir/MERF/baselines/structural-evolution/data/ab_mutagenesis_expts/cr6261/cr6261_exp_data.csv"
cr9114_data_path = "/home/dataset-local/projects_dir/MERF/baselines/structural-evolution/data/ab_mutagenesis_expts/cr9114/cr9114_exp_data.csv"

cr6261_data = pd.read_csv(cr6261_data_path, dtype={'genotype': str})
cr9114_data = pd.read_csv(cr9114_data_path, dtype={'genotype': str})

# 创建各自位置对应的映射
cr6261_mutations = [
    'P28T', 'R30S', 'T58A', 'K59N', 'P62Q', 'D74K', 'F75S', 'A76T', 'G77S', 'V79A', 'V104L'
]
# 论文代码中提供的，感觉是错误的
# cr9114_mutations = [
#     'S29F', 'N30S', 'N31S', 'D46E', 'S52I', 'S57T', 'T58A', 'A59N', 'S71T', 'I74K', 'F75S',
#     'S76T', 'N77S', 'N84S', 'T87R', 'F95Y'
# ]
cr9114_mutations = [
    'S29F', 'N30S', 'N31S', 'S52I', 'S57T', 'T58A', 'A59N', 'S71T', 'I74K', 'F75S',
    'S76T', 'N77S', 'N84S', 'T87R', 'F95Y', 'S106Y'
]

In [5]:
cr9114_data

,genotype,h1_repa,h1_repb,h1_repc,h1_mean,h1_sem,h3_repa,h3_repb,h3_repc,h3_mean,...,pos8,pos9,pos10,pos11,pos12,pos13,pos14,pos15,pos16,som_mut
0,1001110010000101,9.502127,9.499467,9.452164,9.484586,0.011476,6.000000,6.000000,6.000000,6.000000,...,0,1,0,0,0,0,1,0,1,7
1,0001111110011011,9.526711,9.446817,9.319445,9.430991,0.042677,6.000000,6.000000,6.000000,6.000000,...,1,1,0,0,1,1,0,1,1,10
2,1111110010011111,9.522475,9.556415,9.412129,9.497006,0.030798,6.719267,6.789486,6.699885,6.736213,...,0,1,0,0,1,1,1,1,1,12
3,1011111110101110,9.520731,9.557455,9.420888,9.499691,0.028852,6.000000,6.000000,6.000000,6.000000,...,1,1,0,1,0,1,1,1,0,12
4,1111101010000111,9.346955,9.398311,9.396946,9.380737,0.011947,6.000000,6.000000,6.000000,6.000000,...,0,1,0,0,0,0,1,1,1,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65531,0000100010110100,NaN,NaN,NaN,NaN,NaN,6.000000,6.000000,NaN,6.000000,...,0,1,0,1,1,0,1,0,0,5
65532,0010100000110110,NaN,NaN,NaN,NaN,NaN,6.000000,6.000000,NaN,6.000000,...,0,0,0,1,1,0,1,1,0,6
65533,0010100010111000,NaN,NaN,NaN,NaN,NaN,6.000000,6.000000,6.000000,6.000000,...,0,1,0,1,1,1,0,0,0,6
65534,0010100110001000,NaN,NaN,NaN,NaN,NaN,6.000000,6.000000,6.000000,6.000000,...,1,1,0,0,0,1,0,0,0,5


In [6]:
# 检查突变和原始序列的野生型是否匹配
cr9114_h_seq_pdb = "QVQLVQSGAEVKKPGSSVKVSCKSSGGTSNNYAISWVRQAPGQGLDWMGGISPIFGSTAYAQKFQGRVTISADIFSNTAYMELNSLTSEDTAVYFCARHGNYYYYSGMDVWGQGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVEPKSCHHHHHH"
cr9114_l_seq_pdb = "QSALTQPPAVSGTPGQRVTISCSGSDSNIGRRSVNWYQQFPGTAPKLLIYSNDQRPSVVPDRFSGSKSGTSASLAISGLQSEDEAEYYCAAWDDSLKGAVFGGGTQLTVLGQPKAAPSVTLFPPSSEELQANKATLVCLISDFYPGAVTVAWKADSSPVKAGVETTTPSKQSNNKYAASSYLSLTPEQWKSHRSYSCQVTHEGSTVEKTVAPTECS"
cr6261_h_seq_pdb = "EVQLVESGAEVKKPGSSVKVSCKASGGPFRSYAISWVRQAPGQGPEWMGGIIPIFGTTKYAPKFQGRVTITADDFAGTVYMELSSLRSEDTAMYYCAKHMGYQVRETMDVWGKGTTVTVSSASTKGPSVFPLAPSSKSTSGGTAALGCLVKDYFPEPVTVSWNSGALTSGVHTFPAVLQSSGLYSLSSVVTVPSSSLGTQTYICNVNHKPSNTKVDKRVEPKSCDK"
cr6261_l_seq_pdb = "QSVLTQPPSVSAAPGQKVTISCSGSSSNIGNDYVSWYQQLPGTAPKLLIYDNNKRPSGIPDRFSGSKSGTSATLGITGLQTGDEANYYCATWDRRPTAYVVFGGGTKLTVLGAAAGQPKAAPSVTLFPPSSEELQANKATLVCLISDFYPGAVTVAWKADSSPVKAGVETTTPSKQSNNKYAASSYLSLTPEQWKSHRSYSCQVTHEGSTVEKTVAPTECS"

for mutation in cr9114_mutations:
    position = int(mutation[1:-1]) - 1
    wt_aa = mutation[0]
    seq_aa = cr9114_h_seq_pdb[position]
    assert wt_aa == seq_aa, f"CR9114 H chain mutation {mutation} does not match wild-type amino acid {seq_aa} at position {position+1}"
    # print(f"Checking mutation {mutation}: wt_aa={wt_aa}, seq_aa={seq_aa}, position={position+1}")

for mutation in cr6261_mutations:
    position = int(mutation[1:-1]) - 1
    wt_aa = mutation[0]
    seq_aa = cr6261_h_seq_pdb[position]
    assert wt_aa == seq_aa, f"CR6261 H chain mutation {mutation} does not match wild-type amino acid {seq_aa} at position {position+1}"
    # print(f"Checking mutation {mutation}: wt_aa={wt_aa}, seq_aa={seq_aa}, position={position+1}")

In [ ]:
# 处理成我们需要的csv格式并存储

save_dir = "/home/dataset-local/projects_dir/MERF/data/CR"

# cr6261 mutations
pdb_id = "cr6261"
chain_id = "A"
h1_score_list = []
h9_score_list = []
mutate_info_list = []

for idx in range(len(cr6261_data)):
    
    mutation_info = []

    # 获取dataframegenotype，如果值为1，则表示该位置发生了突变，因此0才是反向突变的标志，提取该位置和对应的氨基酸突变信息，添加链信息，加入mutation_info
    genotype = cr6261_data['genotype'].loc[idx]
    for i, bit in enumerate(str(genotype)):
        if bit == '0':
            mutation = cr6261_mutations[i]
            position = int(mutation[1:-1])
            wt_aa = mutation[0]
            mut_aa = mutation[-1]
            mutation_info.append(f"{wt_aa}{chain_id}{position}{mut_aa}")

    mutate_info = ','.join(mutation_info)

    if len(mutation_info) == 0:
        print(genotype)
    
    mutate_info_list.append(mutate_info)
    h1_score_list.append(cr6261_data['h1_mean'].loc[idx])
    h9_score_list.append(cr6261_data['h9_mean'].loc[idx])

pdb_id_list = [f"{pdb_id}_h1"] * len(cr6261_data)
cr6261_h1_output_df = pd.DataFrame({
    'pdb_id': pdb_id_list,
    'mutant': mutate_info_list,
    'h1_score': h1_score_list,
})

save_path = os.path.join(save_dir, f"cr6261_h1.csv")
cr6261_h1_output_df.to_csv(save_path, index=False)

pdb_id_list = [f"{pdb_id}_h9"] * len(cr6261_data)
cr6261_h9_output_df = pd.DataFrame({
    'pdb_id': pdb_id_list,
    'mutant': mutate_info_list,
    'h9_score': h9_score_list
})

save_path = os.path.join(save_dir, f"cr6261_h9.csv")
cr6261_h9_output_df.to_csv(save_path, index=False)

# same for cr9114 mutations
pdb_id = "cr9114"
chain_id = "A"
h1_score_list = []
h3_score_list = []

mutate_info_list = []
for idx in range(len(cr9114_data)):
    
    mutation_info = []

    # 获取dataframegenotype，如果值为1，则表示该位置发生了突变，提取该位置和对应的氨基酸突变信息，添加链信息，加入mutation_info
    genotype = cr9114_data['genotype'].loc[idx]
    for i, bit in enumerate(str(genotype)):
        if bit == '0':
            mutation = cr9114_mutations[i]
            position = int(mutation[1:-1])  # 1-based position
            wt_aa = mutation[0]
            mut_aa = mutation[-1]
            mutation_info.append(f"{wt_aa}{chain_id}{position}{mut_aa}")

    mutate_info = ','.join(mutation_info)

    if len(mutation_info) == 0:
        print(genotype)
    
    mutate_info_list.append(mutate_info)
    h1_score_list.append(cr9114_data['h1_mean'].loc[idx])
    h3_score_list.append(cr9114_data['h3_mean'].loc[idx])

pdb_id_list = [f"{pdb_id}_h1"] * len(cr9114_data)
cr9114_h1_output_df = pd.DataFrame({
    'pdb_id': pdb_id_list,
    'mutant': mutate_info_list,
    'h1_score': h1_score_list
})
save_path = os.path.join(save_dir, f"cr9114_h1.csv")
cr9114_h1_output_df.to_csv(save_path, index=False)

pdb_id_list = [f"{pdb_id}_h3"] * len(cr9114_data)
cr9114_h3_output_df = pd.DataFrame({
    'pdb_id': pdb_id_list,
    'mutant': mutate_info_list,
    'h3_score': h3_score_list
})
save_path = os.path.join(save_dir, f"cr9114_h3.csv")
cr9114_h3_output_df.to_csv(save_path, index=False)

11111111111
1111111111111111


#### 1.4 检查csv数据

In [7]:
# 检查是否有nan

import pandas as pd

csv_dir = "/home/lfj/projects_dir/MERF/data/CR"
csv_list = ["cr6261_h1.csv", "cr6261_h9.csv", "cr9114_h1.csv", "cr9114_h3.csv"]

for csv_file in csv_list:
    csv_path = os.path.join(csv_dir, csv_file)
    df = pd.read_csv(csv_path)
    # 踢掉mutant为nan的行，单独新建一个文件存储
    df_cleaned = df[~df['mutant'].isna()]
    nan_rows = df[df['mutant'].isna()]
    if not nan_rows.empty:
        nan_save_path = os.path.join(csv_dir, f"{csv_file.replace('.csv', '_ref.csv')}")
        nan_rows.to_csv(nan_save_path, index=False)
        print(f"Saved rows with NaN in 'mutant' column to {nan_save_path}")

    df_cleaned.to_csv(csv_path, index=False)

Saved rows with NaN in 'mutant' column to /home/lfj/projects_dir/MERF/data/CR/cr6261_h1_ref.csv
Saved rows with NaN in 'mutant' column to /home/lfj/projects_dir/MERF/data/CR/cr6261_h9_ref.csv
Saved rows with NaN in 'mutant' column to /home/lfj/projects_dir/MERF/data/CR/cr9114_h1_ref.csv
Saved rows with NaN in 'mutant' column to /home/lfj/projects_dir/MERF/data/CR/cr9114_h3_ref.csv


#### 1.5 突变，见scripts

In [7]:
# 突变，见scripts，这里检查突变后文件是否都已存在，同时也作为进度指示器

import pandas as pd
import os
import sys
from tqdm import tqdm

data_path_list = [
    "/home/lfj/projects_dir/MERF/data/CR/cr6261_h1.csv",
    "/home/lfj/projects_dir/MERF/data/CR/cr6261_h9.csv",
    "/home/lfj/projects_dir/MERF/data/CR/cr9114_h1.csv",
    "/home/lfj/projects_dir/MERF/data/CR/cr9114_h3.csv"
]

wt_dir = "/home/lfj/projects_dir/MERF/data/CR/PDBs/"
fix_dir = "/home/lfj/projects_dir/MERF/data/CR/PDBs_fixed/"
mut_dir = "/home/lfj/projects_dir/MERF/data/CR/PDBs_mutated/"

df_list = []
for data_path in data_path_list:
    df_list.append(pd.read_csv(data_path))
df = pd.concat(df_list, ignore_index=True)

not_exist_file_list = []

for idx in tqdm(range(len(df))):
    row = df.iloc[idx]
    pdb_id = row['pdb_id']
    mutant = row['mutant']

    mutate_info_new = mutant

    wt_pdb_file = os.path.join(wt_dir, f"{pdb_id}.pdb")
    mut_pdb_file = os.path.join(mut_dir, f"{pdb_id}_{mutate_info_new}.pdb")

    if not os.path.exists(mut_pdb_file):
        # print(f"Mutated PDB file {mut_pdb_file} does not exist")
        not_exist_file_list.append(mut_pdb_file)

print(f"Total missing mutated PDB files: {len(not_exist_file_list)}")

100%|██████████| 134902/134902 [00:09<00:00, 13862.35it/s]

Total missing mutated PDB files: 0


In [8]:
# 检查突变后的位点是否为想要的突变结果
import os
import pandas as pd
from tqdm import tqdm
from multiprocessing import Pool
from Bio.PDB import PDBParser, PDBIO, Select, Selection

STANDARD_RESIDUE_SUBSTITUTIONS = {
    '2AS':'ASP', '3AH':'HIS', '5HP':'GLU', 'ACL':'ARG', 'AGM':'ARG', 'AIB':'ALA', 'ALM':'ALA',
    'ALA':'ALA', 'ARG':'ARG', 'ASN':'ASN', 'ASP':'ASP', 'CYS':'CYS', 'GLU':'GLU', 'GLN':'GLN',
    'GLY':'GLY', 'HIS':'HIS', 'ILE':'ILE', 'LEU':'LEU', 'LYS':'LYS', 'MET':'MET', 'PHE':'PHE',
    'PRO':'PRO', 'SER':'SER', 'THR':'THR', 'TRP':'TRP', 'TYR':'TYR', 'VAL':'VAL', 'UNK':'UNK', 'MSE':'MET'
}

RESIDUE_NAME_TO_TOKEN = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLU": "E", "GLN": "Q", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V", "UNK": "X",
}

def is_aa(value):
    return value in STANDARD_RESIDUE_SUBSTITUTIONS

def parse_biopython_structure_mapping(pdb_id, entity, max_seq_len=None):
    chains = Selection.unfold_entities(entity, 'C')
    chains.sort(key=lambda c: c.get_id())
    data = {}

    assert len(chains) == 1, f"Expected one chain in antigen entity for {pdb_id}"

    pos_aa_mapping = {}

    for chain in chains:
        residues = Selection.unfold_entities(chain, 'R')
        residues.sort(key=lambda res: (res.get_id()[1], res.get_id()[2]))

        for res in residues:
            res_id = int(res.get_id()[1])
            if max_seq_len is not None and res_id > max_seq_len:
                break

            resname = res.get_resname()
            if not is_aa(resname):
                continue

            std_name = STANDARD_RESIDUE_SUBSTITUTIONS[resname]
            token = RESIDUE_NAME_TO_TOKEN[std_name]

            pos_aa_mapping[res_id] = token

    return pos_aa_mapping

def check_mutation(args):
    """Check mutation for a single row"""
    idx, pdb_id, mutant, mut_dir = args

    pdb_id = pdb_id.replace('+', '')
    pdb_id = pdb_id.replace('.00', '')

    # 3. update mutate_info
    mutate_infos = mutant
    pdb_path = os.path.join(mut_dir, f"{pdb_id}_{mutate_infos}.pdb")
    # 3. end

    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(pdb_id, pdb_path)
        pdb_structure = structure[0]

        for mutate_info in mutate_infos.split(','):
            chain_id = mutate_info[1]

            pos_aa_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[chain_id])

            # Check mutate info
            if pos_aa_mapping.get(int(mutate_info[2:-1]), '') != mutate_info[-1]:
                error_msg = f"Mutation position mismatch in {pdb_id} for mutation {mutate_info}: expected {mutate_info[-1]}, found {pos_aa_mapping.get(int(mutate_info[2:-1]), '')}, all mutations are {mutate_infos}"
                return error_msg
    except Exception as e:
        error_msg = f"Error processing {pdb_id} for mutation {mutant}: {str(e)}"
        return error_msg

    return None

# 1. read data and set path
data_path_list = [
    "/home/lfj/projects_dir/MERF/data/CR/cr6261_h1.csv",
    "/home/lfj/projects_dir/MERF/data/CR/cr6261_h9.csv",
    "/home/lfj/projects_dir/MERF/data/CR/cr9114_h1.csv",
    "/home/lfj/projects_dir/MERF/data/CR/cr9114_h3.csv"
]

wt_dir = "/home/lfj/projects_dir/MERF/data/CR/PDBs/"
fix_dir = "/home/lfj/projects_dir/MERF/data/CR/PDBs_fixed/"
mut_dir = "/home/lfj/projects_dir/MERF/data/CR/PDBs_mutated/"

df_list = []
for data_path in data_path_list:
    df_list.append(pd.read_csv(data_path))
data_df = pd.concat(df_list, ignore_index=True)
# 1. end

# 2. parse mutation info
args_list = [
    (idx, data_df.loc[idx, 'pdb_id'], data_df.loc[idx, 'mutant'], mut_dir)
    for idx in range(len(data_df))
]
# 2. end

# 并行处理（使用 tqdm 显示进度）
with Pool(processes=16) as pool:
    results = list(tqdm(
        pool.imap_unordered(check_mutation, args_list),
        total=len(args_list),
        desc="Checking mutations"
    ))

# 收集并打印错误信息
errors = [msg for msg in results if msg is not None]

if errors:
    print(f"\nFound {len(errors)} mutation errors:")
    for error in errors:
        print(error)
else:
    print(f"\nAll {len(data_df)} mutations are correct!")


Checking mutations: 100%|██████████| 134902/134902 [19:57<00:00, 112.66it/s]


All 134902 mutations are correct!


### 2. SARS dataset in PNAS

In [10]:
# check data
import yaml
import pandas as pd
import os

yml_path = "/home/lfj/projects_dir/MERF/baselines/DiffAffinity/context_generator/configs/inference/7FAE_RBD_Fv_mutation.yml"
with open(yml_path, 'r') as f:
    config = yaml.safe_load(f)

mutations = config['mutations']

494


In [16]:
# 检查mutations的wt是否和pdb序列一致

from Bio import SeqIO
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import PPBuilder

pdb_path = "/home/lfj/projects_dir/MERF/data/7FAE/PDBs/7FAE.pdb"

# get seq of chain H in this pdb
parser = PDBParser()
structure = parser.get_structure('protein_structure', pdb_path)
model = structure[0]
chain = model['H']

ppb = PPBuilder()
sequence_parts = []
for pp in ppb.build_peptides(chain):
    # pp.get_sequence() 返回一个 Seq 对象
    sequence_parts.append(str(pp.get_sequence()))

pdb_seq = ''.join(sequence_parts)

for mutation in mutations:
    position = int(mutation[2:-1]) - 1  # 转换为0-based索引
    wt_aa = mutation[0]
    seq_aa = pdb_seq[position]
    assert wt_aa == seq_aa, f"Mutation {mutation} does not match wild-type amino acid {seq_aa} at position {position+1}"
    # print(f"Checking mutation {mutation}: wt_aa={wt_aa}, seq_aa={seq_aa}, position={position+1}")

/home/lfj/anaconda3/envs/merf/lib/python3.8/site-packages/Bio/PDB/PDBParser.py:395: PDBConstructionWarning: Ignoring unrecognized record 'END' at line 3300
  warnings.warn(


In [ ]:
# 创建我们需要的csv

all_possible_mutations = set("ACDEFGHIKLMNPQRSTVWY")

pdb_id = "7FAE"
mutation_info_list = []
for mutation in mutations:
    wt = mutation[0]
    for mut in all_possible_mutations:
        if mut != wt:
            mutation_info_list.append(mutation.replace('*', mut))

print(len(mutation_info_list))

# create dataframe
pdb_id_list = [pdb_id] * len(mutation_info_list)
mutation_df = pd.DataFrame({
    'pdb_id': pdb_id_list,
    'mutant': mutation_info_list
})

save_dir = "/home/lfj/projects_dir/MERF/data/7FAE/"
save_path = os.path.join(save_dir, f"{pdb_id}.csv")
mutation_df.to_csv(save_path, index=False)

In [10]:
# 突变，见scripts，这里检查突变后文件是否都已存在

import pandas as pd
import os
import sys

if __name__ == '__main__':

    data_path = "/home/lfj/projects_dir/MERF/data/7FAE/7FAE.csv"
    wt_dir = "/home/lfj/projects_dir/MERF/data/7FAE/PDBs/"
    fix_dir = "/home/lfj/projects_dir/MERF/data/7FAE/PDBs_fixed/"
    mut_dir = "/home/lfj/projects_dir/MERF/data/7FAE/PDBs_mutated/"

    df = pd.read_csv(data_path)
    for idx in range(len(df)):
        row = df.iloc[idx]
        pdb_id = row['pdb_id']
        mutant = row['mutant']

        mutate_info_new = mutant

        wt_pdb_file = os.path.join(wt_dir, f"{pdb_id}.pdb")
        fix_pdb_file = os.path.join(fix_dir, f"{pdb_id}.pdb")
        mut_pdb_file = os.path.join(mut_dir, f"{pdb_id}_{mutate_info_new}.pdb")

        if not os.path.exists(fix_pdb_file):
            raise FileNotFoundError(f"Fixed PDB file {fix_pdb_file} does not exist")
        if not os.path.exists(mut_pdb_file):
            print(f"Mutated PDB file {mut_pdb_file} does not exist")


In [1]:
# 检查突变后的位点是否为想要的突变结果
import os
import pandas as pd
from tqdm import tqdm
from multiprocessing import Pool
from Bio.PDB import PDBParser, PDBIO, Select, Selection

STANDARD_RESIDUE_SUBSTITUTIONS = {
    '2AS':'ASP', '3AH':'HIS', '5HP':'GLU', 'ACL':'ARG', 'AGM':'ARG', 'AIB':'ALA', 'ALM':'ALA',
    'ALA':'ALA', 'ARG':'ARG', 'ASN':'ASN', 'ASP':'ASP', 'CYS':'CYS', 'GLU':'GLU', 'GLN':'GLN',
    'GLY':'GLY', 'HIS':'HIS', 'ILE':'ILE', 'LEU':'LEU', 'LYS':'LYS', 'MET':'MET', 'PHE':'PHE',
    'PRO':'PRO', 'SER':'SER', 'THR':'THR', 'TRP':'TRP', 'TYR':'TYR', 'VAL':'VAL', 'UNK':'UNK', 'MSE':'MET'
}

RESIDUE_NAME_TO_TOKEN = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLU": "E", "GLN": "Q", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V", "UNK": "X",
}

def is_aa(value):
    return value in STANDARD_RESIDUE_SUBSTITUTIONS

def parse_biopython_structure_mapping(pdb_id, entity, max_seq_len=None):
    chains = Selection.unfold_entities(entity, 'C')
    chains.sort(key=lambda c: c.get_id())
    data = {}

    assert len(chains) == 1, f"Expected one chain in antigen entity for {pdb_id}"

    pos_aa_mapping = {}

    for chain in chains:
        residues = Selection.unfold_entities(chain, 'R')
        residues.sort(key=lambda res: (res.get_id()[1], res.get_id()[2]))

        for res in residues:
            res_id = int(res.get_id()[1])
            if max_seq_len is not None and res_id > max_seq_len:
                break

            resname = res.get_resname()
            if not is_aa(resname):
                continue

            std_name = STANDARD_RESIDUE_SUBSTITUTIONS[resname]
            token = RESIDUE_NAME_TO_TOKEN[std_name]

            pos_aa_mapping[res_id] = token

    return pos_aa_mapping

def check_mutation(args):
    """Check mutation for a single row"""
    idx, pdb_id, mutant, mut_dir = args

    pdb_id = pdb_id.replace('+', '')
    pdb_id = pdb_id.replace('.00', '')

    # 2. update mutate_info
    mutate_infos = mutant
    pdb_path = os.path.join(mut_dir, f"{pdb_id}_{mutate_infos}.pdb")
    # 3. end

    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(pdb_id, pdb_path)
        pdb_structure = structure[0]

        for mutate_info in mutate_infos.split(','):
            chain_id = mutate_info[1]

            pos_aa_mapping = parse_biopython_structure_mapping(pdb_id, pdb_structure[chain_id])

            # Check mutate info
            if pos_aa_mapping.get(int(mutate_info[2:-1]), '') != mutate_info[-1]:
                error_msg = f"Mutation position mismatch in {pdb_id} for mutation {mutate_info}: expected {mutate_info[-1]}, found {pos_aa_mapping.get(int(mutate_info[2:-1]), '')}, all mutations are {mutate_infos}"
                return error_msg
    except Exception as e:
        error_msg = f"Error processing {pdb_id} for mutation {mutant}: {str(e)}"
        return error_msg

    return None

# 1. read data and set path
data_path = "/home/lfj/projects_dir/MERF/data/7FAE/7FAE.csv"
mut_dir = "/home/lfj/projects_dir/MERF/data/7FAE/PDBs_mutated/"

data_df = pd.read_csv(data_path)

# 1. end

# 2. parse mutation info
args_list = [
    (idx, data_df.loc[idx, 'pdb_id'], data_df.loc[idx, 'mutant'], mut_dir)
    for idx in range(len(data_df))
]
# 2. end

# 并行处理（使用 tqdm 显示进度）
with Pool(processes=16) as pool:
    results = list(tqdm(
        pool.imap_unordered(check_mutation, args_list),
        total=len(args_list),
        desc="Checking mutations"
    ))

# 收集并打印错误信息
errors = [msg for msg in results if msg is not None]

if errors:
    print(f"\nFound {len(errors)} mutation errors:")
    for error in errors:
        print(error)
else:
    print(f"\nAll {len(data_df)} mutations are correct!")


Checking mutations: 100%|██████████| 494/494 [00:03<00:00, 125.98it/s]


All 494 mutations are correct!
